In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
stroke_data = pd.read_csv("stroke-prediction.csv")
X = stroke_data.drop(columns = ["stroke", "id"])
y = stroke_data["stroke"]
names = stroke_data.columns
print(names)

In [ ]:
features = ["age", "avg_glucose_level"]

X2 = StandardScaler().fit_transform(X[features])
Sigma = np.cov(X2, rowvar = False)

print("Covariance matrix:\n ", Sigma.round(2))

fig, ax = plt.subplots(figsize = (4, 3.6))
ax.scatter(X2[:, 0], X2[:, 1], s = 6, alpha = 0.5)
ax.set_xlabel("age (standardised)")
ax.set_ylabel("average glucose level (standardised)")
_ = ax.set_aspect("equal")

In [ ]:
thetas = np.linspace(0, np.pi, 180)
var_z = [np.var(X2 @ np.array([np.cos(t), np.sin(t)]), ddof = 1) for t in thetas]
best = thetas[np.argmax(var_z)]
worst = thetas[np.argmin(var_z)]

print(f"Principle axes 1: {best:.2f}")
print(f"Principal axes 2: {worst:.2f}")
print(f"Therefore, total variance is: {best + worst:.2f}")

fig, axs = plt.subplots(1, 2, figsize = (6, 2.6))
axs[0].plot(np.degrees(thetas), var_z)
axs[0].axvline(np.degrees(best), color = "k", linestyle = ":")
axs[0].axvline(np.degrees(worst), color = "k", linestyle = ":")
axs[0].set_xlabel(r"angle $\theta$ (degrees)")
axs[0].set_ylabel("variance of projection")
axs[1].scatter(X2[:, 0], X2[:, 1], s = 3, alpha = 0.4)
for t, c in [(best, "C1"), (best + np.pi / 2, "C2")]:
    axs[1].plot([-3 * np.cos(t), 3 * np.cos(t)], [-3 * np.sin(t), 3 * np.sin(t)], c)
axs[1].set_aspect("equal")
plt.tight_layout()

In [ ]:
evals, evecs = np.linalg.eigh(Sigma)
order = np.argsort(evals)[::-1]
evals, evecs = evals[order], evecs[:, order]
print("eigenvalues", evals.round(2), " sum:", evals.sum().round(2))
print("eigenvectors (columns):\n", evecs.round(3))
Z2 = X2 @ evecs
print("covariance of the scores:\n", np.cov(Z2, rowvar = False).round(2))

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

In [ ]:
# Convert categorical columns to numeric dummy variables
Xw = pd.get_dummies(X, drop_first=False, dtype=float)

# Convert any missing values, such as N/A in bmi, to the column median
Xw = Xw.apply(pd.to_numeric, errors="coerce")
Xw = Xw.fillna(Xw.median())

feature_names = Xw.columns.to_numpy()

raw = PCA().fit(Xw)
std = make_pipeline(StandardScaler(), PCA()).fit(Xw)

print("explained variance ratio, raw:",
      raw.explained_variance_ratio_[:4].round(3))
print("explained variance ratio, scaled:",
      std[-1].explained_variance_ratio_[:4].round(3))

j = np.argmax(np.abs(raw.components_[0]))

print(f"raw PC1 is dominated by '{feature_names[j]}'"
      f"(variance {Xw.iloc[:, j].var():.0f}; "
      f"next largest {np.sort(Xw.var(axis=0))[-2]:.0f})")

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(5, shuffle=True, random_state = 0)

Zw = std.transform(Xw)

views = [("average glucose level vs age", Xw.iloc[:, :2].to_numpy()), 
        ("PCA, not scaled", raw.transform(Xw)[:, :2]), 
        ("PCA, standardized", Zw[:, :2])]

fig, axs = plt.subplots(1, 3, figsize = (18, 6))
for ax, (title, P2) in zip(axs, views):
    acc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), P2, y, cv = cv).mean()
    for c in range(3):
        ax.scatter(P2[y == c, 0], P2[y == c, 1], s = 6, label = f"class {c}")
    ax.set_title(f"{title}\nCV accuracy {acc:.2f}", fontsize = 9)
    ax.set_xticks([]); ax.set_yticks([])
_ = axs[0].legend(fontsize = 7)

## IGNORE EVERYTHING ABOVE THIS

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, validation_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn import svm

MY_FILE = "stroke-prediction.csv"
raw = pd.read_csv(MY_FILE)

TARGET = "stroke"
DROP = ["id"]

df = raw.drop(columns=DROP)
y = df[TARGET]
X = df.drop(columns=[TARGET])

numeric_cols = X.select_dtypes("number").columns.tolist()
text_cols = [c for c in X.columns if c not in numeric_cols]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), text_cols),
])

C = 1.0  # SVM regularization parameter
models = {
    "SVC linear": svm.SVC(kernel="linear", C=C, class_weight = 'balanced'),
    "LinearSVC": svm.LinearSVC(C=C, max_iter=10000, class_weight = 'balanced'),
    "SVC rbf": svm.SVC(kernel="rbf", gamma=0.7, C=C, class_weight = 'balanced'),
    "SVC poly": svm.SVC(kernel="poly", degree=3, gamma="auto", C=C, class_weight = 'balanced'),
}

Xm_train, Xm_test, ym_train, ym_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

def compare(models, X, y, cv, scoring=("roc_auc", "recall")):
    rows = []
    for name, model in models.items():
        pipe = Pipeline([("prep", preprocess), ("model", model)])
        s = cross_validate(pipe, X, y, cv=cv, scoring=list(scoring), return_train_score=True)
        row = {"model": name}
        for m in scoring:
            row[m] = s["test_" + m].mean()
            row[m + " spread"] = s["test_" + m].std()
        row["training " + scoring[0]] = s["train_" + scoring[0]].mean()
        row["fit ms"] = 1000 * s["fit_time"].mean()
        row["score ms per 1000 rows"] = 1000 * s["score_time"].mean() / (len(X) / cv.get_n_splits()) * 1000
        rows.append(row)
    return pd.DataFrame(rows).set_index("model").round(3)

results = compare(models, Xm_train, ym_train, cv)
results

,roc_auc,roc_auc spread,recall,recall spread,training roc_auc,fit ms,score ms per 1000 rows
model,,,,,,,
SVC linear,0.843,0.032,0.804,0.094,0.854,260.785,89.557
LinearSVC,0.845,0.029,0.824,0.103,0.856,25.151,24.871
SVC rbf,0.704,0.036,0.176,0.064,0.990,193.406,222.867
SVC poly,0.808,0.029,0.643,0.081,0.891,187.411,89.808
